# 05-04 Routing: 조건에 따라 다른 경로로 분기하기

## 학습 목표

이 노트북을 완료하면 다음을 할 수 있어요:

- 입력 데이터를 분류하는 체인을 구성하고, 분류 결과에 따라 다른 처리 경로로 라우팅(Routing)하는 패턴을 구현할 수 있어요
- `RunnableLambda` 방식과 `RunnableBranch` 방식의 차이를 설명하고, 상황에 맞게 선택할 수 있어요
- 분류 체인과 라우팅 체인을 파이프라인으로 연결하여 실제 챗봇 구조를 설계할 수 있어요

## 사전 지식

이 노트북을 시작하기 전에 다음 개념을 알고 있으면 좋아요:

- `RunnableLambda`로 함수를 Runnable로 변환하는 방법
- LCEL 파이프 연산자(`|`) 사용법
- `operator.itemgetter`의 기본 사용법

---

라우팅을 통해 이전 단계의 출력이 다음 단계를 결정하는 동적(Dynamic) 체인을 만들 수 있어요. 이를 통해 LLM과의 상호작용에 구조와 일관성을 제공할 수 있어요.

아래 다이어그램은 이 노트북에서 구현할 조건 분기 흐름이에요:

```mermaid
flowchart TD
    Q["사용자 질문"]:::input
    C["분류 체인<br/>프로그래밍 / 요리 / 기타 판별"]:::process
    R["route() 함수<br/>분류 결과 확인"]:::process
    P["프로그래밍 전문가 체인"]:::process
    K["요리 전문가 체인"]:::process
    G["일반 체인"]:::process
    O["최종 답변"]:::output

    Q --> C --> R
    R -- "프로그래밍" --> P --> O
    R -- "요리" --> K --> O
    R -- "기타" --> G --> O

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

**라우팅 방법:**

1. **`RunnableLambda`에서 조건부로 Runnable 반환** (권장)
2. **`RunnableBranch` 사용**

두 방법 모두 동일한 기능을 제공하지만, LangChain 공식 문서에서는 첫 번째 방법을 권장해요.

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

In [2]:
from operator import itemgetter

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")


## 1. 질문 분류 체인 생성

먼저 입력 질문을 카테고리로 분류하는 체인을 만들어요.

예제에서는 질문을 `프로그래밍`, `요리`, 또는 `기타` 중 하나로 분류해요. 분류 결과가 이후 라우팅의 판단 기준이 되기 때문에, 정확하고 일관된 출력 형식이 중요해요.

> **주의:** 분류 프롬프트에 "한 단어로만 응답하세요" 조건을 포함하는 이유는 이후 `route()` 함수에서 문자열 포함 여부로 분기를 결정하기 때문이에요. 분류 결과가 일관되지 않으면 라우팅이 예상대로 동작하지 않을 수 있어요.

In [4]:
# ============================================================
# TODO: 입력 질문을 프로그래밍/요리/기타로 분류하는 체인을 완성하세요
# 힌트: ChatPromptTemplate.from_template(...) | model | StrOutputParser()
# 예상 결과: "파이썬 질문" → "프로그래밍", "김치찌개" → "요리", "날씨" → "기타"
# ============================================================

# ---------------------------------------------------
# 질문 분류 체인 생성 — 라우팅의 판단 기준
# ---------------------------------------------------
classification_prompt = ChatPromptTemplate.from_template(
  """
  주어진 사용자 질문을 '프로그래밍', '요리', 또는 '기타' 중 하나로 분류하세요. 한 단어로만 응답하세요.
  
  <question>
  {question}
  </question>
  
  Classification:
  """
)

classification_chain = (
  classification_prompt
  | model
  | StrOutputParser()
)


In [5]:
# 분류 체인 테스트
test_questions = [
    "파이썬에서 리스트와 튜플의 차이는?",
    "김치찌개 만드는 방법은?",
    "오늘 날씨는 어때?"
]

classification_chain.invoke(test_questions)

'프로그래밍'

## 2. 카테고리별 전문 체인 생성

각 카테고리에 맞는 전문 프롬프트를 가진 체인을 만들어요.

각 체인은 시스템 역할을 다르게 설정하여 해당 분야에 특화된 답변을 제공해요.

In [6]:
# ============================================================
# TODO: 각 카테고리에 맞는 전문 체인 3개(프로그래밍/요리/일반)를 정의하세요
# 힌트: ChatPromptTemplate.from_template(...) | model (파서 생략 가능)
# 예상 결과: 각 체인이 역할에 맞는 프롬프트로 응답
# ============================================================

# ---------------------------------------------------
# 카테고리별 전문 체인 — 역할 프롬프트(System Prompt) 분리
# ---------------------------------------------------
# "프로그래밍 전문가로서..."로 시작하도록 강제
programming_chain = (
    ChatPromptTemplate.from_template(
        """당신은 프로그래밍 전문가입니다.
항상 답변을 "프로그래밍 전문가로서..."로 시작하세요.

질문: {question}
답변:"""
    )
    | model
)

# 2단계: 요리 전문 체인
cooking_chain = (
    ChatPromptTemplate.from_template(
        """당신은 요리 전문가입니다.
항상 답변을 "요리 전문가로서..."로 시작하세요.

질문: {question}
답변:"""
    )
    | model
)

# 3단계: 일반 체인 (기타 카테고리용 기본 응답)
general_chain = (
    ChatPromptTemplate.from_template(
        """다음 질문에 간결하게 답변해주세요:

질문: {question}
답변:"""
    )
    | model
)

## 3. RunnableLambda를 이용한 라우팅 (권장 방법)

LangChain 공식 문서에서 권장하는 방식이에요.

사용자 정의 함수를 `RunnableLambda`로 래핑하여 조건에 따라 적절한 체인을 반환해요. 이 방법은 파이썬 표준 조건문으로 로직을 작성할 수 있어 가독성이 좋아요.

In [10]:
# ============================================================
# TODO: route() 함수를 작성하고 RunnableLambda로 감싸서 전체 체인을 구성하세요
# 힌트: def route(info): if "프로그래밍" in info["topic"]: return programming_chain ...
#       full_chain = {"topic": classification_chain, "question": itemgetter("question")} | RunnableLambda(route) | ...
# 예상 결과: 분류 결과에 따라 적절한 전문 체인으로 라우팅
# ============================================================
from langchain_core.runnables import RunnablePassthrough
# ---------------------------------------------------
# RunnableLambda 라우팅 — 분류 결과로 체인 선택
# ---------------------------------------------------
def route(info: dict):
    topic = info.get("topic", "").lower()

    if "프로그래밍" in topic:
        return programming_chain
    elif "요리" in topic:
        return cooking_chain
    else:
        return general_chain

full_chain = (
    # {
    #   "topic": classification_chain,
    #   "question": itemgetter("question")
    # }
    RunnablePassthrough.assign(topic=classification_chain)
    | RunnableLambda(route)
    | StrOutputParser()
)

In [11]:
# 프로그래밍 관련 질문 테스트
res1 = full_chain.invoke({"question": "파이썬에서 딕셔너리와 리스트의 차이는 뭐여?"})
res1

"프로그래밍 전문가로서... 딕셔너리와 리스트는 파이썬에서 매우 중요한 자료구조이며, 각각의 특성과 용도가 다릅니다.\n\n1. **리스트(List)**:\n   - 순서가 있는 값들의 집합입니다.\n   - 요소들은 인덱스를 통해 접근하며, 인덱스는 0부터 시작합니다.\n   - 같은 타입의 데이터뿐만 아니라 서로 다른 타입의 데이터도 저장할 수 있습니다.\n   - 중복된 값을 허용합니다.\n   - 예: `my_list = [1, 2, 3, 'hello', 3]` \n\n2. **딕셔너리(Dictionary)**:\n   - 키-값 쌍으로 이루어진 집합입니다.\n   - 각 키는 유일하며, 키를 통해서 값에 접근합니다.\n   - 정렬된 순서를 유지하지 않지만, 파이썬 3.7 이상에서는 삽입 순서를 기억합니다.\n   - 키는 불변형 데이터 타입만 사용할 수 있으며, 값은 어떤 데이터 타입도 가능합니다.\n   - 예: `my_dict = {'name': 'Alice', 'age': 30, 'city': 'New York'}` \n\n이 두 자료구조는 그 사용 목적에 따라 선택할 수 있으며, 리스트는 순서가 중요한 데이터 작업에, 딕셔너리는 키를 통해 데이터를 빠르게 조회하고자 할 때 유용합니다."

In [12]:
# 요리 관련 질문 테스트
res2 = full_chain.invoke({"question": "라면 끓일때 면이랑 스프중 뭘 먼저 넣는게 맛있어?"})
res2

'요리 전문가로서... 라면을 끓일 때는 먼저 면을 넣고 끓이는 것이 좋습니다. 면을 끓인 후, 면이 적당히 익었을 때 스프를 추가하면 스프의 맛이 면에 잘 배어 더욱 깊은 풍미를 느낄 수 있습니다. 또한, 물이 끓기 시작한 후 약간의 시간을 두고 면을 넣으면, 면이 더 쫄깃하게 익는데 도움이 됩니다. 그래서 이 방법을 추천드립니다!'

In [ ]:
# 기타 질문 테스트


## 4. RunnableBranch 사용법

`RunnableBranch`는 입력값에 따라 실행할 조건과 Runnable을 정의할 수 있는 특별한 유형의 `Runnable`이에요.

**문법:**
- `RunnableBranch`는 `(조건, Runnable)` 쌍의 리스트와 기본 Runnable로 초기화해요
- 호출 시 전달된 입력값을 각 조건에 전달하여 분기를 선택해요
- True로 평가되는 첫 번째 조건을 선택하고, 해당 Runnable을 실행해요
- 모든 조건이 False이면 기본 `Runnable`을 실행해요

> **실무 팁:** `RunnableLambda` 방식은 파이썬 표준 조건문으로 복잡한 로직을 표현할 수 있어요. `RunnableBranch`는 단순한 조건 분기에 적합하지만, 조건이 복잡해질수록 `RunnableLambda` 방식이 더 유지보수하기 쉬워요.

In [15]:
# ============================================================
# TODO: RunnableBranch를 사용하여 동일한 라우팅 로직을 구현하세요
# 힌트: RunnableBranch((조건_람다, 체인), ..., 기본_체인)
# 예상 결과: RunnableLambda 방식과 동일한 라우팅 결과
# ============================================================

# ---------------------------------------------------
# RunnableBranch: (조건, Runnable) 튜플 목록으로 분기 정의
# ---------------------------------------------------
branch = RunnableBranch(
    (lambda x: "프로그래밍" in x["topic"].lower(), programming_chain),
    (lambda x: "요리" in x["topic"].lower(), cooking_chain),
    general_chain
)

full_chain_branch = (
  RunnablePassthrough.assign(topic=classification_chain)
  | branch
  | StrOutputParser()
)

In [16]:
# RunnableBranch 체인 테스트
res = full_chain_branch.invoke({"question": "자바스크립트와 자바의 차이점은?"})
res

'프로그래밍 전문가로서, 자바스크립트와 자바의 차이점은 여러 가지가 있습니다. 가장 첫 번째로, 두 언어는 서로 다른 목적과 환경에서 사용됩니다. 자바는 주로 서버 사이드 애플리케이션, 안드로이드 앱 개발, 그리고 엔터프라이즈 솔루션에서 사용되는 객체 지향 프로그래밍 언어입니다. 반면, 자바스크립트는 주로 웹 개발에서 클라이언트 사이드 스크립팅에 사용되며, 브라우저에서 실행됩니다.\n\n두 번째로, 문법과 언어 구조도 다릅니다. 자바는 강타입 언어로, 명확한 타입 선언이 필요하며, 클래스 기반의 객체 지향 프로그래밍을 따릅니다. 자바스크립트는 느슨한 타이핑을 갖는 동적 언어로, 함수형 및 객체 지향 프로그래밍 패러다임을 지원합니다.\n\n마지막으로, 실행 환경 차이도 있습니다. 자바는 JVM(Java Virtual Machine)에서 실행되는 반면, 자바스크립트는 웹 브라우저 또는 Node.js와 같은 런타임 환경에서 실행됩니다. 이러한 차이점들은 두 언어의 사용 사례 및 기능에도 큰 영향을 미칩니다.'

## 마무리 요약

### 라우팅 방법 비교

| 항목 | `RunnableLambda` (권장) | `RunnableBranch` |
|------|------------------------|------------------|
| 코드 구조 | 일반 파이썬 함수 | 튜플 리스트 |
| 복잡한 조건 처리 | 용이 | 제한적 |
| 가독성 | 높음 | 보통 |
| LangChain 권장 | 예 | 아니요 |

### 핵심 요점

- 라우팅은 입력 분류 결과에 따라 서로 다른 체인으로 전달하는 패턴이에요
- 분류 체인의 출력 형식이 일관되어야 라우팅 로직이 안정적으로 동작해요
- `RunnableLambda`로 `route()` 함수를 래핑하면 일반 파이썬 조건문으로 분기 로직을 표현할 수 있어요
- `RunnableBranch`는 기능적으로 동일하지만, LangChain은 `RunnableLambda` 방식을 권장해요

### 다음 노트북 예고

다음 노트북(`05-RunnableParallel.ipynb`)에서는 여러 체인을 동시에 실행하여 처리 속도를 높이는 `RunnableParallel` 패턴을 배워요. 순차 실행과 병렬 실행의 성능 차이를 직접 측정해볼게요.